# 🤖 **Model Benchmark — BTC Target + Top ETH Cross-Asset Features (selective)**

### 🎯 **Objective**
Re-test the BTC + ETH cross-asset hypothesis with a **restrictive selection** of the top 5 ETH features only, to avoid dilution from lower-signal features.

### 🔎 **Context**
`07_model_benchmark_btc_with_eth.ipynb` used all 17 engineered ETH features → PR-AUC ≈ 0.766, essentially equal to the 17-feature BTC baseline (0.771).  
The old `02_model_benchmark_btc_eth.ipynb` used ~5-10 prioritised ETH features → PR-AUC 0.777 (best BTC result).  
Hypothesis: **the top 3-5 ETH features carry real signal; features ranked 6-17 add noise that cancels the gain.**

### 📊 **Feature Set**

| Source | Features | Score (combined EDA 06) | Count |
|---|---|---|---|
| BTC market (notebook 01 EDA) | 17 standard features | — | 17 |
| ETH cross — **top 5** (EDA 06) | `momentum_volatility_eth` (1.145), `return_7d_eth` (1.120), `lag_return_7d_eth` (0.894), `volatility_30d_eth` (0.670), `volatility_7d_eth` (0.663) | ≥ 0.663 | 5 |
| **Total** | | | **22** |

Three variants defined in the feature selection cell:
- **top5** (score ≥ 0.663) — used by default
- **top7** (score ≥ 0.567) — adds `lag_volatility_7d_eth`, `lag_volume_norm_eth`
- **top10** (score ≥ 0.414) — adds `pressure_x_return_eth`, `volume_norm_eth`, `return_1d_eth`

### 🚀 **Goal**
Determine whether focused ETH cross-features outperform both the BTC baseline (01) and the full-ETH version (07).

In [ ]:
import pandas as pd
from pipelines.ml_benchmark import run_benchmark_pipeline

### **1. Load BTC and ETH Price Data**

In [ ]:
df_btc = pd.read_csv("../../data/gold/market/btc_usdt_1d_features.csv")
df_eth = pd.read_csv("../../data/gold/market/eth_usdt_1d_features.csv")

print(f"BTC price : {df_btc.shape[0]} rows, {df_btc.shape[1]} columns")
print(f"ETH price : {df_eth.shape[0]} rows, {df_eth.shape[1]} columns")

### **2. Merge BTC + ETH (inner join)**

In [ ]:
df_btc["open_time"] = pd.to_datetime(df_btc["open_time"], errors="coerce")
df_eth["open_time"] = pd.to_datetime(df_eth["open_time"], errors="coerce")

eth_renamed = df_eth.rename(columns={c: f"{c}_eth" for c in df_eth.columns if c != "open_time"})

df = pd.merge(df_btc, eth_renamed, on="open_time", how="inner")
df.drop(columns=["target_eth", "open_time"], inplace=True)

print(f"Merged dataset : {df.shape[0]} rows, {df.shape[1]} columns")

### **3. Feature Selection — Top ETH cross-features only**

Combined scores from `06_eda_btc_with_eth.ipynb` (|Spearman| + MI, normalized against all features):

| Variant | ETH features | Score threshold |
|---|---|---|
| top5 | `momentum_volatility_eth`, `return_7d_eth`, `lag_return_7d_eth`, `volatility_30d_eth`, `volatility_7d_eth` | ≥ 0.663 |
| top7 | top5 + `lag_volatility_7d_eth`, `lag_volume_norm_eth` | ≥ 0.567 |
| top10 | top7 + `pressure_x_return_eth`, `volume_norm_eth`, `return_1d_eth` | ≥ 0.414 |

In [ ]:
# ── ETH cross-feature variants (by combined score from EDA 06) ──────────────
ETH_TOP5 = [
    "momentum_volatility_eth",  # 1.145
    "return_7d_eth",            # 1.120
    "lag_return_7d_eth",        # 0.894
    "volatility_30d_eth",       # 0.670
    "volatility_7d_eth",        # 0.663
]

ETH_TOP7 = ETH_TOP5 + [
    "lag_volatility_7d_eth",    # 0.608
    "lag_volume_norm_eth",      # 0.567
]

ETH_TOP10 = ETH_TOP7 + [
    "pressure_x_return_eth",    # 0.553
    "volume_norm_eth",          # 0.515
    "return_1d_eth",            # 0.497
]

# ── Change this to ETH_TOP7 or ETH_TOP10 to test other variants ─────────────
ETH_CROSS = ETH_TOP5

SELECTED_FEATURES = [
    # BTC market features (17)
    "return_1d", "return_7d",
    "volatility_7d", "volatility_30d",
    "buy_pressure",
    "drawdown",
    "ma_ratio",
    "lag_return_1d", "lag_return_7d",
    "lag_volatility_7d",
    "lag_buy_pressure",
    "lag_volume_norm",
    "momentum_acc",
    "momentum_volatility",
    "volume",
    "number_of_trades",
    "quote_asset_volume",
] + ETH_CROSS

TARGET = "target"

df = df[SELECTED_FEATURES + [TARGET]]

print(f"Dataset shape : {df.shape}  ({len(ETH_CROSS)} ETH cross-features)")
print(f"ETH features  : {ETH_CROSS}")
print(f"Target distribution:\n{df[TARGET].value_counts()}")
df.isna().sum().loc[lambda s: s > 0]

### **4. Model Benchmark**

In [ ]:
results = run_benchmark_pipeline(df, verbose=False, plot_confusion=True)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("pr_auc", ascending=False)

results_df.loc["mean"] = results_df.select_dtypes("number").mean()
results_df.loc["std"]  = results_df.select_dtypes("number").std()

results_df

### **5. Conclusion**

Compare against:
- `01_model_benchmark_btc.ipynb` — BTC baseline: CatBoost **0.771**, LightGBM 0.752, XGB 0.762, RF 0.744
- `07_model_benchmark_btc_with_eth.ipynb` — 17 ETH cross: CatBoost 0.766, LightGBM **0.764**, XGB 0.762, RF 0.735

Key question: does restricting to top 5 ETH features (score ≥ 0.663) recover the gain visible in the old NB02 (0.777)?

If top5 > 01 → selective cross-asset works → test top7/top10 by changing `ETH_CROSS` above.  
If top5 ≈ 01 → the ETH cross-asset signal doesn't translate to prediction gain regardless of selection.  
If top5 < 01 → even the best ETH features hurt BTC — ETH adds no cross-asset value.